In [ ]:
import pandas as pd
from quotaclimat.data_processing.factiva.explo_optimal_thresholds.utils_explo.llm_classif_secteur import (
    classify_dataframe_secteur_async,
    llm_gpt_4_1,
)

from quotaclimat.data_processing.factiva.explo_optimal_thresholds.utils_explo.utils_explo import filter_env_articles
DATA_PATH = 'quotaclimat/data_processing/factiva/explo_optimal_thresholds/data_explo/factiva_keywords.xlsx'
EXPORT_PATH = 'quotaclimat/data_processing/factiva/explo_optimal_thresholds/data_explo/factiva_sectors.xlsx'

In [ ]:
df = pd.read_excel(DATA_PATH).drop(columns=["Unnamed: 0"], errors="ignore")

In [ ]:
df['text'] = (
    df['title'].fillna('') + ' ' +
    df['snippet'].fillna('') + ' ' +
    df['body'].fillna('')
)

df['llm_secteurs'] = None
df['llm_secteur_evidences'] = None

In [ ]:
df = filter_env_articles(df)

In [ ]:
out_df_4_1 = await classify_dataframe_secteur_async(
    df.loc[df.llm_secteurs.isna()].reset_index(drop=True).iloc[:500][[col for col in df.columns if col not in ['llm_secteurs', 'llm_secteur_evidences']]], text_col="text", llm=llm_gpt_4_1, batch_size=2
)

In [ ]:
valid_llm = out_df_4_1[out_df_4_1['llm_secteurs'] != ""].copy()
valid_llm = (
    valid_llm[["an", "llm_secteurs", "llm_secteur_evidences"]]
    .drop_duplicates(subset="an", keep="last")
    .set_index("an")
)

In [ ]:
for col in ["llm_secteurs", "llm_secteur_evidences"]:
    mask = df[col].isna()
    df.loc[mask, col] = df.loc[mask, "an"].map(valid_llm[col])

In [ ]:
df[(df.llm_secteurs != "") & ~(df.llm_secteurs.isna())]

In [ ]:
df[(df.llm_secteurs != "") & ~(df.llm_secteurs.isna())].to_excel(EXPORT_PATH, sheet_name='test_dataset')